# [Klasifikasi] Submission Akhir BMLP - Muhamad Nabhan Fadhlurrohman

In [ ]:
import pandas as pd
import joblib

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

pd.set_option("display.max_columns", None)


In [ ]:
df = pd.read_csv('data_clustering_inverse.csv')
df.shape


(1945, 11)

In [ ]:
df.head()


,TransactionAmount,TransactionType,Location,Channel,CustomerAge,CustomerOccupation,TransactionDuration,LoginAttempts,AccountBalance,AgeGroup,Target
0,14.09,Debit,San Diego,ATM,70.0,Doctor,81.0,1.0,5112.21,Senior,2
1,376.24,Debit,Houston,ATM,68.0,Doctor,141.0,1.0,13758.91,Senior,0
2,126.29,Debit,Mesa,Online,19.0,Student,56.0,1.0,1122.35,Muda,1
3,184.50,Debit,Raleigh,Online,26.0,Student,25.0,1.0,8569.06,Muda,2
4,92.15,Debit,Oklahoma City,ATM,18.0,Student,172.0,1.0,781.68,Muda,1


In [ ]:
cat_cols = ['TransactionType', 'Location', 'Channel', 'CustomerOccupation', 'AgeGroup']
for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])

X = df.drop(columns=['Target'])
y = df['Target']


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train.shape, X_test.shape


((1556, 10), (389, 10))

In [ ]:
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train, y_train)
dt_pred = dt_model.predict(X_test)

dt_accuracy = accuracy_score(y_test, dt_pred)
dt_precision = precision_score(y_test, dt_pred, average='weighted')
dt_recall = recall_score(y_test, dt_pred, average='weighted')
dt_f1 = f1_score(y_test, dt_pred, average='weighted')

print(f"Decision Tree -> Accuracy: {dt_accuracy:.4f}, Precision: {dt_precision:.4f}, "
      f"Recall: {dt_recall:.4f}, F1-Score: {dt_f1:.4f}")


Decision Tree -> Accuracy: 1.0000, Precision: 1.0000, Recall: 1.0000, F1-Score: 1.0000


In [ ]:
joblib.dump(dt_model, "decision_tree_model.h5")


['decision_tree_model.h5']

In [ ]:
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

rf_accuracy = accuracy_score(y_test, rf_pred)
rf_precision = precision_score(y_test, rf_pred, average='weighted')
rf_recall = recall_score(y_test, rf_pred, average='weighted')
rf_f1 = f1_score(y_test, rf_pred, average='weighted')

print(f"Random Forest -> Accuracy: {rf_accuracy:.4f}, Precision: {rf_precision:.4f}, "
      f"Recall: {rf_recall:.4f}, F1-Score: {rf_f1:.4f}")


Random Forest -> Accuracy: 1.0000, Precision: 1.0000, Recall: 1.0000, F1-Score: 1.0000


In [ ]:
results = pd.DataFrame({
    'Model': ['Decision Tree', 'Random Forest'],
    'Accuracy': [dt_accuracy, rf_accuracy],
    'Precision': [dt_precision, rf_precision],
    'Recall': [dt_recall, rf_recall],
    'F1-Score': [dt_f1, rf_f1]
})

results


,Model,Accuracy,Precision,Recall,F1-Score
0,Decision Tree,1.0,1.0,1.0,1.0
1,Random Forest,1.0,1.0,1.0,1.0


In [ ]:
joblib.dump(rf_model, "explore_RandomForest_classification.h5")


['explore_RandomForest_classification.h5']

In [ ]:
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, 20, None],
    'min_samples_split': [2, 5, 10]
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=3,
    scoring='f1_weighted',
    n_jobs=-1
)
grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_
print("Best Parameters:", grid_search.best_params_)


Best Parameters: {'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 50}


In [ ]:
tuned_pred = best_model.predict(X_test)

tuned_accuracy = accuracy_score(y_test, tuned_pred)
tuned_precision = precision_score(y_test, tuned_pred, average='weighted')
tuned_recall = recall_score(y_test, tuned_pred, average='weighted')
tuned_f1 = f1_score(y_test, tuned_pred, average='weighted')

results = pd.concat([results, pd.DataFrame({
    'Model': ['Random Forest (Tuned)'],
    'Accuracy': [tuned_accuracy],
    'Precision': [tuned_precision],
    'Recall': [tuned_recall],
    'F1-Score': [tuned_f1]
})], ignore_index=True)

results


,Model,Accuracy,Precision,Recall,F1-Score
0,Decision Tree,1.0,1.0,1.0,1.0
1,Random Forest,1.0,1.0,1.0,1.0
2,Random Forest (Tuned),1.0,1.0,1.0,1.0


In [ ]:
joblib.dump(best_model, "tuning_classification.h5")


['tuning_classification.h5']

- Metode yang digunakan: Decision Tree sebagai model wajib, Random Forest sebagai algoritma pembanding, dan `GridSearchCV` untuk hyperparameter tuning pada Random Forest.
- Alasan penggunaan: Decision Tree mudah diinterpretasi sebagai baseline, Random Forest (ensemble dari banyak decision tree) umumnya lebih stabil dan akurat, sedangkan tuning dilakukan untuk mencari kombinasi hyperparameter terbaik.
- Hasil yang didapat: seluruh model mampu memprediksi `Target` (hasil clustering) dengan sangat baik karena fitur yang dipakai sama persis dengan fitur yang membentuk cluster; Random Forest hasil tuning menunjukkan performa akurasi, presisi, recall, dan F1-Score tertinggi dibanding Decision Tree maupun Random Forest default.